# Uniprot Usage Demo
This document demonstrates how to use the Uniprot API implementation programmatically.

This api permits searching via 3 ways:
- **Stream search**: This method is suitable for text queries. It uses the same search syntax as the Uniprot website.
- **ID search**: This method is suitable for searching specific entries by their Uniprot IDs for a given pandas DataFrame.
- **Sequence search**: This method is suitable for searching entries by their protein sequences. It uses the BLAST algorithm to find similar sequences.


In [2]:
from bioseq_dl import UniprotInterface
import pandas as pd

## Stream Search

### Define the query
- Query: Should be defined as a string containing the search criteria. The query syntax is the same as the one used in the Uniprot website.
- Fields: List of entry sections to be returned. More fields can be found in the [Uniprot documentation](https://rest.uniprot.org/configure/uniprotkb/result-fields).
- Sort: Specify field by wich to sort results.

In [ ]:
query="organism_name:homo sapiens (human) AND length:[15 TO 30] AND reviewed:true"
fields="accession,protein_name,sequence,ec,lineage,organism_name,ft_mutagen,ft_variant,ft_domain,ft_motif,ft_region,ft_act_site,ft_binding,ft_site,xref_pfam,xref_alphafolddb,xref_pdb,go_id"
sort="accession asc"

### Instantiating the API

In [37]:
instance = UniprotInterface(
    total_retries=5
)

### Making the request

In [38]:
response = instance.submit_stream(
    query=query,
    fields=fields,
    sort=sort,
    include_isoform=True,
    download=False,
    format="json"
)

### Parsing results

In [41]:
instance.parse_stream_response(
    query=query,
    response=response,
    extract_fields=None
).head(5)

,query,accession,protein_name,organism_name,taxon_id,ineage,sequence,length,alphafold_ids,biogrid_ids,...,interpro_ids,kegg_ids,panther_ids,pathwaycommons_ids,pdb_ids,pfam_ids,pride_ids,reactome_ids,refseq_ids,string_ids
0,organism_name:homo sapiens (human) AND length:...,A0A075B6S0,T cell receptor gamma joining 1,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",NYYKKLFGSGTTLVVT,16,[A0A075B6S0],[],...,[],[],[],[],[],[],[],[],[],[]
1,organism_name:homo sapiens (human) AND length:...,A0A075B6Y3,T cell receptor alpha joining 3,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",GYSSASKIIFGSGTRLSIRP,20,[A0A075B6Y3],[],...,[],[],[],[],[],[],[],[],[],[]
2,organism_name:homo sapiens (human) AND length:...,A0A075B6Y9,T cell receptor alpha joining 42,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",YGGSQGNLIFGKGTKLSVKP,20,[A0A075B6Y9],[],...,[],[],[],[],[],[],[],[],[],[]
3,organism_name:homo sapiens (human) AND length:...,A0A075B700,T cell receptor alpha joining 31,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",NNNARLMFGDGTQLVVKP,18,[A0A075B700],[],...,[],[],[],[],[],[],[],[],[],[]
4,organism_name:homo sapiens (human) AND length:...,A0A075B706,T cell receptor delta joining 1,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",TDKLIFGKGTRVTVEP,16,[A0A075B706],[],...,[],[],[],[],[],[],[],[],[],[]


## ID Search

### Define the query
A query has to be defined. This query should contain:
- Identifiers: A list of Uniprot IDs to be fetched.
- From db: The database from which the IDs originate. In this case, it is "UniProtKB_AC-ID".
- To db: The target database to which the IDs will be mapped. In this case,
it is "UniProtKB".

In [22]:
df = pd.DataFrame({
    "ids": ["P05067"]
})
from_db = "UniProtKB_AC-ID"
to_db = "UniProtKB"

### Instantiating the API

In [23]:
instance = UniprotInterface(
    total_retries=5
)

### Making the request

In [27]:
response = instance.download_batch(
    dataset=df,
    column_ids="ids",
    auto_db=False,
    from_db=from_db,
    to_db=to_db,
    batch_size=5
)

Processing manual IDs: 100%|██████████ 1/1 [00:03<00:00,  3.78s/it] Processing manual IDs

Fetched: 1 / 1


### Parsing results

Wether if we dont want to filter the results and get all the available fields, we can set `extract_fields=None`.

In [34]:
instance.parse_results(response, extract_fields=None)

,accession,protein_name,organism_name,gene_primary,taxon_id,ineage,sequence,length,alphafold_ids,biogrid_ids,...,pfam_ids,pride_ids,reactome_ids,refseq_ids,rhea_ids,string_ids,references,features,keywords,source_db
0,P05067,Amyloid-beta precursor protein,Homo sapiens,[APP],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,770,[P05067],[106848],...,"[PF10515, PF12924, PF12925, PF02177, PF03494, ...",[],"[R-HSA-114608, R-HSA-3000178, R-HSA-381426, R-...","[NP_000475.1, NP_001129488.1, NP_001129601.1, ...",[],[9606.ENSP00000284981],[{'title': 'The precursor of Alzheimer's disea...,"[{'type': 'Signal', 'description': '', 'locati...","[3D-structure, Alternative splicing, Alzheimer...",unknown


If we want to filter the results and get specific fields, we can set `extract_fields` to a list of desired fields. For example, to get the accession number, ID, protein name, gene names, organism name, and length, we can set `extract_fields` as follows:

In [29]:
instance.parse_results(response, extract_fields=["accession", "id", "protein_name", "gene_names", "organism_name", "length"])

,accession,protein_name,organism_name,length,source_db
0,P05067,Amyloid-beta precursor protein,Homo sapiens,770,unknown


## Sequence Search
For this search type it will need a list of sequences to be searched.

In [3]:
df = pd.read_csv("data/unknown_sequences.csv")
df

,sequence
0,MAFSDLTSRTVHLYDNWIKDADPRVEDWLLMSSPLPQTILLGFYVY...
1,MAGQHLPVPRLEGVSREQFMQHLYPQRKPLVLEGIDLGPCTSKWTV...


### Preparing BLAST
For this search type we will need to import the blast module from the bioseqdownloader package.

In [4]:
from bioseq_dl.core.utils.blast_search import (
    download_uniprot_database,
    check_blast,
    make_blast_database,
    run_blast,
    parse_blast_results
)

First we need to check if BLAST is installed in the system. If not, we need to install it.

Then, we need to download the Uniprot database in FASTA format.

In [5]:
check_blast()

2025-10-27 11:52:24 | INFO     | bioseq_dl.core.utils.blast_search | System-wide BLAST is installed.


'/home/diego/micromamba/envs/bioseqdownloader/bin/blastp'

In [7]:
download_uniprot_database("uniprotkb_reviewed", extension="fasta")
make_blast_database("uniprotkb_reviewed", extension="fasta")

2025-10-27 11:57:28 | INFO     | bioseq_dl.core.utils.blast_search | Database uniprotkb_reviewed already exists at /home/diego/.cache/bioseq_dl/blast_db/uniprotkb_reviewed.fasta.
2025-10-27 11:57:28 | INFO     | bioseq_dl.core.utils.blast_search | BLAST database already exists at /home/diego/.cache/bioseq_dl/blast_db/uniprotkb_reviewed. No need to create it again.


### Running BLAST
This will create an output file named "tmp/blast_results.txt".

In [8]:
sequences = df["sequence"].dropna().tolist()
run_blast(
    sequences=sequences,
    db_name="uniprotkb_reviewed",
    blast_type="blastp",
    evalue=1e-5,
)

2025-10-27 12:27:38 | INFO     | bioseq_dl.core.utils.blast_search | Running BLAST search...


After that, we can parse the results and get a pandas DataFrame with the results.

In [11]:
results = parse_blast_results("tmp/blast_results.txt")
results_df = pd.DataFrame(results)
results_df.head()

,query,subject,identity,alignment_length,evalue,bit_score
0,0,sp|A1L3X0|ELOV7_HUMAN,100.000,281,0,0
1,0,sp|A0JNC4|ELOV7_BOVIN,91.103,281,25,0
2,1,sp|A2RUC4|TYW5_HUMAN,100.000,315,0,0


Now you can manipulate the DataFrame as needed. For example, you can rename columns, drop unnecessary columns, and extract specific information from the subject IDs.

In [13]:
df_blast = results_df.rename(columns={"query": "id", "subject": "subject_id"})
df_blast = df_blast.drop(columns=["id"])
df_blast["accession"] = df_blast["subject_id"].apply(lambda x: x.split("|")[1])
df_blast = df_blast.drop(columns=["subject_id","alignment_length", "evalue", "bit_score"])
df_blast

,identity,accession
0,100.000,A1L3X0
1,91.103,A0JNC4
2,100.000,A2RUC4


After done, you can do a search in the Uniprot database using the accession numbers obtained from the BLAST results to get more information about the sequences.

In [15]:
instance = UniprotInterface()
results = instance.download_batch(
    dataset=df_blast,
    column_ids="accession",
    auto_db=False,
    from_db="UniProtKB_AC-ID",
    to_db="UniProtKB",
    batch_size=10
)
final_df = instance.parse_results(results, extract_fields=["accession", "id", "protein_name", "sequence", "gene_names", "organism_name", "length"])
final_df

Processing manual IDs:   0%|           0/3 [00:00<?, ?it/s] Processing manual IDs

2025-10-27 12:53:15 | INFO     | bioseq_dl.interfaces.uniprot | Fetched: 3 / 3


Processing manual IDs: 100%|██████████ 3/3 [00:03<00:00,  1.14s/it] Processing manual IDs


,accession,protein_name,organism_name,sequence,length,source_db
0,A1L3X0,Very long chain fatty acid elongase 7,Homo sapiens,MAFSDLTSRTVHLYDNWIKDADPRVEDWLLMSSPLPQTILLGFYVY...,281,unknown
1,A0JNC4,Very long chain fatty acid elongase 7,Bos taurus,MAFSDLTSRTVRLYDNWIKDADPRVEDWLLMSSPLPQTIILGFYVY...,281,unknown
2,A2RUC4,tRNA wybutosine-synthesizing protein 5,Homo sapiens,MAGQHLPVPRLEGVSREQFMQHLYPQRKPLVLEGIDLGPCTSKWTV...,315,unknown
